# Loading

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import os

In [ ]:
import json
with open('Output/GRN_GPU_output/midbrain/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)
grn_dict.keys()

In [ ]:
adata = sc.read_h5ad("Process_Data/cellbin_3D_region/combined_adata_midbrain.h5ad")
adata

# Data process

In [ ]:
import pandas as pd
import anndata
from tqdm.auto import tqdm
from typing import Dict, List, Optional
import omicverse as ov
import os
import warnings

def compute_and_concat_adata_by_group(
    adata: anndata.AnnData,
    group_key: str,
    pathway_dict: Dict[str, List[str]],
    output_dir: str,  
    num_workers: int = 30
) -> anndata.AnnData:  

    per_group_results_adatas = []
    unique_groups = list(adata.obs[group_key].cat.categories)

    for g in unique_groups:
        sub_adata = adata[adata.obs[group_key] == g].copy()
        if sub_adata.n_obs == 0:
            continue
            
        sub_adata_processed: anndata.AnnData = ov.single.pathway_aucell_enrichment(
                sub_adata,
                pathways_dict=pathway_dict,
                num_workers=num_workers
            )

        safe_g_name = str(g).replace('/', '_').replace(' ', '_')
        output_filename = os.path.join(output_dir, f"auc_anndata_group_{safe_g_name}.h5ad")

        sub_adata_processed.write_h5ad(output_filename, compression='gzip')

        per_group_results_adatas.append(sub_adata_processed)

    
    adata_all = anndata.concat(per_group_results_adatas, axis=0)

    original_cell_order = adata.obs_names
    cells_in_result = adata_all.obs_names
    final_cell_order = [cell for cell in original_cell_order if cell in cells_in_result]
    
    final_adata = adata_all[final_cell_order].copy()
    
    print("AUCell 计算、保存和 AnnData 拼接完成。")
    return final_adata

In [ ]:
auc_mtx = compute_and_concat_adata_by_group(adata,'celltype',grn_dict,
                              output_dir='Process_Data/cellbin_3D_region/auc_adata',num_workers=3)

In [ ]:
auc_mtx.write_h5ad('Process_Data/cellbin_3D_region/auc_mtx.h5ad', compression='gzip')

In [ ]:
adata

In [ ]:
auc_mtx = sc.read_h5ad("Process_Data/cellbin_3D_region/auc_mtx.h5ad")
auc_mtx.obs['celltype'] = adata[auc_mtx.obs_names,:].obs['celltype']
auc_mtx.var['mean'] = auc_mtx.to_df().mean(axis=0)
auc_mtx = auc_mtx[:,auc_mtx.var['mean']!=0]
auc_mtx

In [ ]:
auc_mtx.var_names

In [ ]:
auc_mtx.var

In [ ]:
sc.pl.matrixplot(
    auc_mtx,
    auc_mtx.var_names,
    'celltype',
    dendrogram=True,
    standard_scale='var',
    vmin=-1,
    vmax=1,
    cmap="RdBu_r",
)

In [ ]:
from pyscenic.rss import regulon_specificity_scores
rss_cellType = regulon_specificity_scores(auc_mtx.to_df(), auc_mtx.obs['celltype'])
rss_cellType

In [ ]:
rss_cellType.T.sort_values(['OPC'])